# Bike Regression — ทำนายจำนวนจักรยานที่ถูกเช่า

In [ ]:
import sys
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(REPO))
import config
print("RANDOM_STATE =", config.RANDOM_STATE)

RANDOM_STATE = 42


## 1. ปัญหา / เป้าหมาย
- TODO: ทำนาย `Rented Bike Count` (คัน/ชม.) — ใครได้ประโยชน์ ใช้ตัดสินใจอะไร

## 2. แหล่งข้อมูล
- TODO: Seoul Bike Sharing Demand (UCI 560, CC BY 4.0) — จำนวนแถว/คอลัมน์จริง

## 3. สำรวจ / เตรียมข้อมูล
- TODO: missing, distribution, ความสัมพันธ์กับ target
- ⚠️ ตัดสินใจจาก train เท่านั้น

### 3.1 ตรวจ encoding และรูปแบบวันที่ของไฟล์ดิบ
**จุดประสงค์**
1. หา encoding ที่อ่านไฟล์ได้ถูกต้อง เพราะชื่อคอลัมน์มีสัญลักษณ์ `°` (เช่น `Temperature(°C)`) ถ้าเลือก encoding ผิด จะอ่านไฟล์ไม่ได้
2. หารูปแบบของคอลัมน์ `Date` (เช่น `01/12/2017`) ว่าเป็น วัน/เดือน หรือ เดือน/วัน ถ้าเลือกผิด วันที่จะสลับกัน และการแบ่งข้อมูลตามวันในขั้นถัดไปจะผิดตาม

In [ ]:
import pandas as pd

# (ก) ดู byte ดิบของหัวตาราง ตรงคอลัมน์อุณหภูมิ
with open(config.BIKE_RAW_CSV, "rb") as f:
    header_bytes = f.readline()
i = header_bytes.find(b"Temperature")
print("bytes ของหัวคอลัมน์อุณหภูมิ:", header_bytes[i:i + 16])

# (ข) ลองอ่านหัวตารางด้วย 2 encoding
for enc in ["utf-8", "cp1252"]:
    try:
        cols = pd.read_csv(config.BIKE_RAW_CSV, encoding=enc, nrows=0).columns
        print(f"{enc} -> อ่านได้:", cols[3])
    except UnicodeDecodeError as e:
        print(f"{enc} -> อ่านไม่ได้:", e.reason)

# (ค) ตรวจรูปแบบวันที่: แยก "01/12/2017" เป็น 3 ส่วน แล้วดูค่าสูงสุดของแต่ละส่วน
dates = pd.read_csv(config.BIKE_RAW_CSV, encoding="cp1252", usecols=["Date"])["Date"] 
parts = dates.str.split("/", expand=True).astype(int)
print("ส่วนที่ 1 ค่าสูงสุด:", parts[0].max())
print("ส่วนที่ 2 ค่าสูงสุด:", parts[1].max())
print("ส่วนที่ 3 ค่าที่มี:", sorted(parts[2].unique().tolist()))

# (ง) แปลงเป็นวันที่ด้วยรูปแบบ วัน/เดือน/ปี แล้วตรวจช่วงวันที่และจำนวนวัน
parsed = pd.to_datetime(dates, format="%d/%m/%Y")
print("ช่วงวันที่:", parsed.min().date(), "ถึง", parsed.max().date())
print("จำนวนวัน:", parsed.nunique(), "| จำนวนแถว:", len(parsed))

bytes ของหัวคอลัมน์อุณหภูมิ: b'Temperature(\xb0C),'
utf-8 -> อ่านไม่ได้: invalid start byte
cp1252 -> อ่านได้: Temperature(°C)
ส่วนที่ 1 ค่าสูงสุด: 31
ส่วนที่ 2 ค่าสูงสุด: 12
ส่วนที่ 3 ค่าที่มี: [2017, 2018]
ช่วงวันที่: 2017-12-01 ถึง 2018-11-30
จำนวนวัน: 365 | จำนวนแถว: 8760


**ผลลัพธ์**
- `°` ถูกเก็บเป็น byte `\xb0`: อ่านด้วย `utf-8` ไม่ได้ แต่อ่านด้วย `cp1252` ได้ถูกต้อง → ใช้ `cp1252`
- วันที่ส่วนแรกมีค่าถึง 31 ส่วนที่สองไม่เกิน 12 → รูปแบบเป็น วัน/เดือน/ปี (`%d/%m/%Y`)
- ข้อมูลตั้งแต่ 1 ธ.ค. 2017 ถึง 30 พ.ย. 2018 มี 365 วัน × 24 ชม. = 8,760 แถว ไม่มีชั่วโมงไหนหาย
- บันทึกทั้งสองค่าไว้ใน `config.py`

## 4. แบ่งข้อมูล
- TODO: โหลด `bike-regression/data/splits/` (แบ่งตามวัน)
- ⚠️ **ห้ามใช้ test ฝึกหรือเลือกโมเดล**

In [ ]:
# TODO

## 5. Train + เหตุผล
- TODO: preprocessing + model ใน `Pipeline` เดียว
- TODO: เลือกโมเดลด้วย `GroupKFold` (group = วันที่) บน train
- TODO: เลือก input ของแอป **≤ 8 feature** + เหตุผล (ผู้ใช้กรอกได้จริง)

In [ ]:
# TODO

## 6. ประเมินบน test + แปลความหมาย
- เมตริก: **MAE (หลัก, คัน/ชม.)**, RMSE, R²
- ⚠️ MAPE ใช้ตรงๆ ไม่ได้ เพราะมียอด = 0
- TODO: แปลความหมาย MAE เทียบกับยอดเช่าเฉลี่ย

In [ ]:
# TODO

## 7. ตัวอย่างผลทำนาย + error analysis
- TODO: ตัวอย่างทำนาย 3–5 แถว · residual vs ชั่วโมง/ฤดู · ช่วงที่ผิดมาก

In [ ]:
# TODO

## 8. บันทึกโมเดล → `app/model.joblib`
- TODO: `joblib.dump(pipeline, config.BIKE_MODEL)` — ทั้ง Pipeline ไฟล์เดียว
- ⚠️ เช็กขนาดไฟล์ < 100 MB · ลองโหลดกลับแล้วทำนายซ้ำให้ได้ค่าเดิม

In [ ]:
# TODO